In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from pybaseball import statcast, playerid_lookup

In [2]:
START_DATE = '2023-03-30'
END_DATE = '2026-09-27'
TEST_START_DATE = '2026-03-25'
CACHE_DIR = Path('data')
CACHE_PATH = CACHE_DIR / f'statcast_{START_DATE}_{END_DATE}.parquet'

if CACHE_PATH.exists():
    raw = pd.read_parquet(CACHE_PATH)
    print(f'Loaded {len(raw):,} pitches from {CACHE_PATH}')
else:
    CACHE_DIR.mkdir(parents = True, exist_ok = True)
    raw = statcast(START_DATE, END_DATE)
    raw.to_parquet(CACHE_PATH, index = False)
    print(f'Downloaded and cached {len(raw):,} pitches to {CACHE_PATH}')

Loaded 3,018,395 pitches from data/statcast_2023-03-30_2026-09-27.parquet


In [3]:
bbe = {'field_out', 'single', 'double', 'home_run', 'force_out', 'grounded_into_double_play',
       'field_error', 'sac_fly', 'triple', 'fielders_choice', 'double_play', 'fielders_choice_out',
       'sac_fly_double_play', 'triple_play'}

In [4]:
contact = raw.loc[
    (raw.events.isin(bbe)) & (raw.game_type == 'R'),
    ['game_date', 'batter', 'events', 'hc_x', 'hc_y', 'launch_speed', 
     'launch_angle', 'stand', 'p_throws', 'pitch_type']]

contact['spray_angle'] = np.degrees(
    np.arctan2(contact["hc_x"] - 125.42, 198.27 - contact["hc_y"])
)

contact['is_hit'] = contact["events"].isin(
    {"single", "double", "triple", "home_run"}
).astype(int)

contact.head()

,game_date,batter,events,hc_x,hc_y,launch_speed,launch_angle,stand,p_throws,pitch_type,spray_angle,is_hit
3,2026-09-27,811965,field_out,106.88,84.45,90.1,45,R,R,SL,-9.251587,0
4,2026-09-27,695720,field_out,167.48,128.36,68.5,46,R,R,SL,31.032406,0
10,2026-09-27,664774,field_out,124.99,187.49,64.4,-53,L,L,ST,-2.284242,0
13,2026-09-27,665161,field_out,149.99,64.47,88.4,30,R,L,FF,10.405429,0
19,2026-09-27,678489,field_out,118.37,209.96,79.6,82,R,L,FF,-148.90668,0


In [5]:
model_data = contact.dropna(
    subset=["launch_speed", "launch_angle", "spray_angle", 
            "stand", "p_throws", "pitch_type", "is_hit"]
).copy()

print(f"Contact events: {len(contact):,}")
print(f"Complete rows: {len(model_data):,}")
print(f"Observed batting average: {model_data['is_hit'].mean():.3f}")
print("Missing values by field:")
print(contact.isna().sum().sort_values(ascending=False))

Contact events: 495,813
Complete rows: 493,896
Observed batting average: 0.324
Missing values by field:
launch_speed    1659
launch_angle    1358
hc_x             211
hc_y             211
spray_angle      211
pitch_type       187
game_date          0
batter             0
events             0
stand              0
p_throws           0
is_hit             0
dtype: int64


In [6]:
model_data.game_date = pd.to_datetime(model_data.game_date)
test_start_date = pd.Timestamp(TEST_START_DATE)
train_data = model_data[model_data['game_date'] < test_start_date].copy()
test_data = model_data[model_data['game_date'] >= test_start_date].copy()

In [7]:
numeric_features = ['launch_speed', 'launch_angle', 'spray_angle']
categorical_features = ['stand', 'p_throws', 'pitch_type']
features = numeric_features + categorical_features

X_train = train_data[features]
X_test = test_data[features]
y_train = train_data['is_hit']
y_test = test_data['is_hit']

print(f"Training rows: {len(train_data):,}")
print(f"Test rows: {len(test_data):,}")
print(f"Training dates: {train_data['game_date'].min().date()} to {train_data['game_date'].max().date()}")
print(f"Test dates: {test_data['game_date'].min().date()} to {test_data['game_date'].max().date()}")

Training rows: 370,575
Test rows: 123,321
Training dates: 2023-03-30 to 2025-09-28
Test dates: 2026-03-25 to 2026-09-27


In [8]:
#Baseline model
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy = 'prior')
baseline.fit(X_train, y_train)
baseline_probs = baseline.predict_proba(X_test)[:, 1]

In [9]:
#Simple Logistic Regression model
#One-hot encoding for categorical features

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss

preprocessor = ColumnTransformer(
    transformers = [
        ('numeric', StandardScaler(), numeric_features),
        ('categorical', OneHotEncoder(handle_unknown = 'ignore'), categorical_features)
    ]
)

logistic_model = Pipeline(
    steps = [
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(max_iter = 1000, random_state = 99))
    ]
)

logistic_model.fit(X_train, y_train)
logistic_model_probs = logistic_model.predict_proba(X_test)[:, 1]

metrics = pd.DataFrame({
    'log_loss': [
        log_loss(y_test, baseline_probs),
        log_loss(y_test, logistic_model_probs)
    ],

    'brier_score': [
        brier_score_loss(y_test, baseline_probs),
        brier_score_loss(y_test, logistic_model_probs)
    ],

    'accuracy_at_0.5': [
        accuracy_score(y_test, baseline_probs >= 0.5),
        accuracy_score(y_test, logistic_model_probs >= 0.5)
    ]
}, index = ['Baseline Classifier', 'Logistic Model'])

metrics

,log_loss,brier_score,accuracy_at_0.5
Baseline Classifier,0.628287,0.218276,0.678141
Logistic Model,0.595752,0.201992,0.689039


In [14]:
#Spline Logistic Model with Interaction Effects

from sklearn.preprocessing import PolynomialFeatures, SplineTransformer

numeric_pipeline = Pipeline(
    steps = [
        ('splines', SplineTransformer(n_knots = 5, degree = 3)),
        ('interactions', PolynomialFeatures(interaction_only = True, include_bias = False))
    ]
)

preprocessor = ColumnTransformer(
    transformers = [
        ('numeric', numeric_pipeline, numeric_features),
        ('categorical', OneHotEncoder(handle_unknown = 'ignore'), categorical_features)
    ]
)

spline_model = Pipeline(
    steps = [
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(max_iter = 1000, random_state = 99))
    ]
)

spline_model.fit(X_train, y_train)
spline_model_probs = spline_model.predict_proba(X_test)[:, 1]

metrics.loc['Spline Logistic Classifier'] = [
    log_loss(y_test, spline_model_probs),
    brier_score_loss(y_test, spline_model_probs),
    accuracy_score(y_test, spline_model_probs >= 0.5)
]

metrics

,log_loss,brier_score,accuracy_at_0.5
Baseline Classifier,0.628287,0.218276,0.678141
Logistic Model,0.595752,0.201992,0.689039
Spline Logistic Classifier,0.430178,0.139634,0.798680


In [16]:
#Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier

rf_preprocessor = ColumnTransformer(
    transformers = [
        ('interactions', PolynomialFeatures(interaction_only = True, include_bias = False), numeric_features),
        ('onehot', OneHotEncoder(handle_unknown = 'ignore'), categorical_features)
    ]
)

rf_model = Pipeline(
    steps = [
        ('preprocessor', rf_preprocessor),
        ('model', RandomForestClassifier(n_estimators = 300))
    ]
)

rf_model.fit(X_train, y_train)
rf_model_probs = rf_model.predict_proba(X_test)[:, 1]

metrics.loc['Random Forest Classifier'] = [
    log_loss(y_test, rf_model_probs),
    brier_score_loss(y_test, rf_model_probs),
    accuracy_score(y_test, rf_model_probs >= 0.5)
]

metrics

,log_loss,brier_score,accuracy_at_0.5
Baseline Classifier,0.628287,0.218276,0.678141
Logistic Model,0.595752,0.201992,0.689039
Spline Logistic Classifier,0.430178,0.139634,0.798680
Random Forest Classifier,0.353494,0.107015,0.847950
